In [247]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.features.feature_engineer.apm_features import _detect_star_players
from src.utils.helper_functions import findOpp
from src.utils.team_info import * 


In [248]:
pd.set_option('display.max_columns', None)

s24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S24.csv').sort_values(by='GAME_DATE')
p24 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P24.csv').sort_values(by='GAME_DATE')
s24 = pd.concat([s24, p24])
s24 = _detect_star_players(s24)

s25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

In [ ]:
# def _build_rotowire_long(rotowire_df: pd.DataFrame) -> pd.DataFrame:
#     df = rotowire_df.copy()

#     # Split "AWAY @ HOME"
#     df[['AWAY_TEAM', 'HOME_TEAM']] = (
#         df['Game'].astype(str).str.split(r'\s*@\s*', n=1, expand=True)
#     )
#     df['AWAY_TEAM'] = df['AWAY_TEAM'].str.strip().str.upper()
#     df['HOME_TEAM'] = df['HOME_TEAM'].str.strip().str.upper()

#     # Tipoff has no year. Rotowire 'Season' = season start year.
#     # Months Oct/Nov/Dec -> Season; Jan-Sep -> Season + 1
#     month_str = df['Tipoff'].astype(str).str.split().str[0]
#     is_first_half = month_str.isin(['Oct', 'Nov', 'Dec'])
#     inferred_year = np.where(is_first_half, df['Season'], df['Season'] + 1)

#     parsed = pd.to_datetime(
#         df['Tipoff'].astype(str) + ' ' + inferred_year.astype(str),
#         format='%b %d %I:%M %p %Y',
#         errors='coerce',
#     )
#     df['GAME_DATE'] = parsed.dt.normalize()

#     # Score "AWAY-HOME"
#     score = df['Score'].astype(str).str.split('-', n=1, expand=True)
#     df['AWAY_SCORE'] = pd.to_numeric(score[0], errors='coerce')
#     df['HOME_SCORE'] = pd.to_numeric(score[1], errors='coerce')

#     home = pd.DataFrame({
#         'GAME_DATE':         df['GAME_DATE'],
#         'TEAM_ABBREVIATION': df['HOME_TEAM'],
#         'TEAM_SPREAD':    df['Home_Line'],
#         'GAME_TOTAL':     df['Over_Under'],
#     })
#     away = pd.DataFrame({
#         'GAME_DATE':         df['GAME_DATE'],
#         'TEAM_ABBREVIATION': df['AWAY_TEAM'],
#         'TEAM_SPREAD':    -df['Home_Line'],
#         'GAME_TOTAL':     df['Over_Under'],
#     })
#     return pd.concat([home, away], ignore_index=True).dropna(subset=['GAME_DATE', 'TEAM_ABBREVIATION'])


# def merge_rotowire(player_df: pd.DataFrame, rotowire_long: pd.DataFrame) -> pd.DataFrame:
#     out = player_df.copy()
#     out['GAME_DATE'] = pd.to_datetime(out['GAME_DATE'], errors='coerce').dt.normalize()
#     out = out.merge(rotowire_long, on=['GAME_DATE', 'TEAM_ABBREVIATION'], how='left')

#     # Fall back to TEAM_SPREAD_ODDS when rotowire is missing (NaN) or 0/-0.
#     # A 0 in the result is only kept if TEAM_SPREAD_ODDS is also 0.
#     if 'TEAM_SPREAD_ODDS' in out.columns:
#         needs_fill = out['TEAM_SPREAD'].isna() | (out['TEAM_SPREAD'] == 0)
#         out.loc[needs_fill, 'TEAM_SPREAD'] = out.loc[needs_fill, 'TEAM_SPREAD_ODDS']
#         out['TEAM_SPREAD'] = out['TEAM_SPREAD'] + 0.0  # collapse -0.0 -> 0.0
#     if 'GAME_TOTAL_ODDS' in out.columns:
#         out['GAME_TOTAL'] = out['GAME_TOTAL'].fillna(out['GAME_TOTAL_ODDS'])

#     return out


# ro25 = pd.read_csv('rotowire_nba_2025.csv')
# ro25_long = _build_rotowire_long(ro25)

# s26 = merge_rotowire(s26, ro25_long)
# p26 = merge_rotowire(p26, ro25_long)

In [249]:
df = pd.concat([s25,s26])
df['STARTING'] = df['START_POSITION'].notna().astype(int)
df['PTS_PER_MIN'] = df['PTS'] / df['MIN'].replace(0,np.nan)
df['AST_PER_MIN'] = df['AST'] / df['MIN'].replace(0,np.nan)
df['REB_PER_MIN'] = df['REB'] / df['MIN'].replace(0,np.nan)
df['IS_HOME'] = df['MATCHUP'].str.contains('vs', na=False).astype(int)
df.sample(10)

,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,name,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME
20830,20829.0,5462.0,2024-25,1630595,Cade Cunningham,Cade,1610612765,DET,Detroit Pistons,22400941,2025-03-11,DET vs. WAS,W,31.533333,11,23,0.478,1,5,0.200,4,5,0.800,1,7,8,10,3,1,0,4,5,5,27,16,51.6,1,0,48.0,1,31:32,1,118.6,120.6,120.6,98.3,98.5,98.5,20.3,22.1,22.1,0.476,3.33,26.3,0.026,0.184,0.105,7.9,7.9,0.500,0.536,0.337,0.339,103.75,102.75,85.62,102.75,0.197,68,11.0,23.0,G,4.40,2.44,5.0,9.0,13.0,88.0,1.0,2.0,56.0,5.0,13.0,0.385,6.0,10.0,0.600,1.0,3.0,0.333,48,104,0.462,16,38,0.421,11,15,0.733,18,43,61,29,12.0,7,5,4,18,18,123,20.0,117.6,119.4,101.2,100.0,16.4,19.4,0.604,2.42,19.2,0.339,0.759,0.553,0.117,0.538,0.556,103.2,103.0,85.83,103,0.598,1610612764,WAS,Washington Wizards,37,92,0.402,13,39,0.333,16,20,0.800,12,35,47,20,13.0,7,4,5,18,18,103,-20.0,101.2,100.0,117.6,119.4,-16.4,-19.4,0.541,1.54,14.6,0.241,0.661,0.447,0.126,0.473,0.511,103.2,103.0,85.83,103,0.402,0,PG,23.0,-14.5,234.5,-15.0,235.0,Cade Cunningham,Jalen Duren,Jaden Ivey,1,1,2,1,NaN,1,0.856237,0.317125,0.253700,1
7133,NaN,19466.0,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,22500337,2025-12-04,MIN @ NOP,W,34.800000,6,14,0.429,0,2,0.000,2,2,1.000,2,5,7,5,2,2,2,0,4,2,14,6,39.9,0,0,34.0,1,34:48,1,115.7,116.3,116.3,109.0,108.8,108.8,6.7,7.5,7.5,0.185,2.50,22.7,0.071,0.125,0.103,9.1,9.1,0.429,0.470,0.200,0.200,110.46,110.34,91.95,110.34,0.084,80,6.0,14.0,F,4.47,2.43,2.0,8.0,10.0,52.0,2.0,0.0,35.0,2.0,6.0,0.333,4.0,8.0,0.500,6.0,10.0,0.600,42,92,0.457,15,38,0.395,26,29,0.897,12,37,49,30,20.0,5,10,2,22,22,125,9.0,110.9,111.6,105.0,105.5,5.8,6.2,0.714,1.50,19.1,0.308,0.811,0.562,0.179,0.538,0.597,111.6,111.0,92.50,112,0.514,1610612740,NOP,New Orleans Pelicans,43,95,0.453,7,25,0.280,23,26,0.885,8,35,43,29,12.0,11,2,10,22,22,116,-9.0,105.0,105.5,110.9,111.6,-5.8,-6.2,0.674,2.42,19.5,0.189,0.692,0.438,0.109,0.489,0.545,111.6,111.0,92.50,110,0.486,0,PF,25.0,NaN,NaN,-11.5,233.5,Anthony Edwards,Julius Randle,Ayo Dosunmu,0,0,2,1,NaN,1,0.402299,0.143678,0.201149,0
6130,NaN,20601.0,2025-26,1628396,Tony Bradley,Tony,1610612754,

In [251]:
player_name = 'Cade Cunningham'
stat_name = 'MIN'

pdf = df[df['PLAYER_NAME'] == player_name].sort_values(by='GAME_DATE')
games_played = pdf.shape[0]
home_stats = pdf[pdf['IS_HOME'] == 1][stat_name]
length_home_stats = len(home_stats)
away_stats = pdf[pdf['IS_HOME'] == 0][stat_name]
length_away_stats = len(away_stats)
active_star_count_0 = pdf[pdf['ACTIVE_STARS_COUNT'] == 0]
length_active_star_count_0 = len(active_star_count_0)
active_star_count_1 = pdf[pdf['ACTIVE_STARS_COUNT'] == 1]
length_active_star_count_1 = len(active_star_count_1)
active_star_count_2 = pdf[pdf['ACTIVE_STARS_COUNT'] == 2]
length_active_star_count_2 = len(active_star_count_2)
active_star_count_3 = pdf[pdf['ACTIVE_STARS_COUNT'] == 3]
length_active_star_count_3 = len(active_star_count_3)
first_20_games = pdf[stat_name].iloc[:20].mean()
last_20_games = pdf[stat_name].iloc[-20:].mean()
last_3_games = pdf[stat_name].iloc[-3:].mean()
heavy_favorite_team = pdf[pdf['TEAM_SPREAD'] < -10]
length_heavy_favorite_team = len(heavy_favorite_team)
favorite_team       = pdf[(pdf['TEAM_SPREAD'] >= -10) & (pdf['TEAM_SPREAD'] < 0)]
length_favorite_team = len(favorite_team)
underdog_team       = pdf[(pdf['TEAM_SPREAD'] > 0) & (pdf['TEAM_SPREAD'] <= 10)]
length_underdog_team = len(underdog_team)
heavy_underdog_team = pdf[pdf['TEAM_SPREAD'] > 10]
length_heavy_underdog_team = len(heavy_underdog_team)
favorite = pdf[pdf['TEAM_SPREAD'] < 0]
length_favorite = len(favorite)
underdog = pdf[pdf['TEAM_SPREAD'] > 0]
length_underdog = len(underdog)
slow_game = pdf[pdf['GAME_TOTAL'] < 225.0]
length_slow_game = len(slow_game)
normal_game = pdf[(pdf['GAME_TOTAL'] >= 225.0) & (pdf['GAME_TOTAL'] <= 235.0)]
length_normal_game = len(normal_game)
fast_game = pdf[pdf['GAME_TOTAL'] > 235.0]
length_fast_game = len(fast_game)


print(f"{pdf['PLAYER_NAME'].iloc[0]} | AVG {stat_name}: {pdf[stat_name].mean().round(2)} | Games Played: {games_played}")
print(f"Last 20 games: {last_20_games.round(2)}")
print(f"Last 3 games: {last_3_games.round(2)}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Home: {home_stats.mean().round(2)} | Games: {length_home_stats}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Away: {away_stats.mean().round(2)} | Games: {length_away_stats}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 0: {round(active_star_count_0[stat_name].mean(), 2)} | Games: {length_active_star_count_0}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 1: {round(active_star_count_1[stat_name].mean(), 2)} | Games: {length_active_star_count_1}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 2: {round(active_star_count_2[stat_name].mean(), 2)} | Games: {length_active_star_count_2}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Active Star Count 3: {round(active_star_count_3[stat_name].mean(), 2)} | Games: {length_active_star_count_3}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Favorite Team: {round(favorite_team[stat_name].mean(), 2)} | Games: {length_favorite_team}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Heavy Favorite Team: {round(heavy_favorite_team[stat_name].mean(), 2)} | Games: {length_heavy_favorite_team}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Underdog Team: {round(underdog_team[stat_name].mean(), 2)} | Games: {length_underdog_team}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Heavy Underdog Team: {round(heavy_underdog_team[stat_name].mean(), 2)} | Games: {length_heavy_underdog_team}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Favorite: {round(favorite[stat_name].mean(), 2)} | Games: {length_favorite}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Underdog: {round(underdog[stat_name].mean(), 2)} | Games: {length_underdog}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Slow Game: {round(slow_game[stat_name].mean(), 2)} | Games: {length_slow_game}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Normal Game: {round(normal_game[stat_name].mean(), 2)} | Games: {length_normal_game}")
print(f"{pdf['PLAYER_NAME'].iloc[0]} Fast Game: {round(fast_game[stat_name].mean(), 2)} | Games: {length_fast_game}")


Cade Cunningham | AVG MIN: 34.98 | Games Played: 145
Last 20 games: 33.23
Last 3 games: 41.25
Cade Cunningham Home: 35.27 | Games: 69
Cade Cunningham Away: 34.73 | Games: 76
Cade Cunningham Active Star Count 0: nan | Games: 0
Cade Cunningham Active Star Count 1: 27.84 | Games: 1
Cade Cunningham Active Star Count 2: 34.98 | Games: 106
Cade Cunningham Active Star Count 3: 35.19 | Games: 38
Cade Cunningham Favorite Team: 35.81 | Games: 66
Cade Cunningham Heavy Favorite Team: 30.84 | Games: 22
Cade Cunningham Underdog Team: 35.62 | Games: 51
Cade Cunningham Heavy Underdog Team: 35.46 | Games: 5
Cade Cunningham Favorite: 34.57 | Games: 88
Cade Cunningham Underdog: 35.6 | Games: 56
Cade Cunningham Slow Game: 35.87 | Games: 59
Cade Cunningham Normal Game: 34.06 | Games: 69
Cade Cunningham Fast Game: 35.58 | Games: 16


In [ ]:
def player_scenarios(df: pd.DataFrame, player_name: str, stat_name: str) -> dict:
    """
    Historical splits for a player covering the context signals
    that the base model does not capture:
        1. Active stars count  (roster context)
        2. Opponent pace       (game-speed context)
        3. Spread              (game-script / blowout risk)
        4. Home / Away         (venue context)

    Bayesian shrinkage toward the player's overall median:
        shrunk = (n * split_median + k * overall_median) / (n + k)
    k is a pseudo-count tuned per split type.
    """
    pdf = df[df['PLAYER_NAME'] == player_name].sort_values(by='GAME_DATE')
    overall_median = pdf[stat_name].median()
    total_n = len(pdf)

    def split_stats(subset, k=10):
        n = len(subset)
        if n == 0:
            return {'median': None, 'shrunk_median': None, 'delta': 0.0, 'hit_rate_vs_overall': None, 'n': 0}
        split_median = subset[stat_name].median()
        shrunk = (n * split_median + k * overall_median) / (n + k)
        hit_rate = (subset[stat_name] >= overall_median).mean()
        return {
            'median':              round(split_median, 4),
            'shrunk_median':       round(shrunk, 4),
            'delta':               round(shrunk - overall_median, 4),
            'hit_rate_vs_overall': round(hit_rate, 4),
            'n':                   n,
        }

    K_ACTIVE_STARS = 10
    K_PACE         = 10
    K_SPREAD       = 10
    K_HOME_AWAY    = 5

    # 1. Active stars count
    active_stars = {
        i: split_stats(pdf[pdf['ACTIVE_STARS_COUNT'] == i], k=K_ACTIVE_STARS)
        for i in [0, 1, 2, 3]
    }

    # 2. Game pace (proxied by Vegas total -- p25 / p75 of the merged dataset)
    opp_pace = {
        'high_pace':   split_stats(pdf[pdf['GAME_TOTAL'] > 234.5], k=K_PACE),
        'middle_pace': split_stats(pdf[(pdf['GAME_TOTAL'] >= 225.0) & (pdf['GAME_TOTAL'] <= 235.0)], k=K_PACE),
        'low_pace':    split_stats(pdf[pdf['GAME_TOTAL'] < 225.0], k=K_PACE),
    }

    # 3. Spread
    spread = {
        'favorite':       split_stats(pdf[(pdf['TEAM_SPREAD'] < 0)], k=K_SPREAD),
        'underdog':       split_stats(pdf[(pdf['TEAM_SPREAD'] > 0)], k=K_SPREAD),
    }

    # 4. Home / Away
    home_away = {
        'home': split_stats(pdf[pdf['IS_HOME'] == 1], k=K_HOME_AWAY),
        'away': split_stats(pdf[pdf['IS_HOME'] == 0], k=K_HOME_AWAY),
    }

    overall_iqr = (
        pdf[stat_name].quantile(0.75) - pdf[stat_name].quantile(0.25)
        if total_n > 0 else None
    )

    return {
        'player':         player_name,
        'stat':           stat_name,
        'overall_median': round(overall_median, 4) if pd.notna(overall_median) else None,
        'overall_iqr':    round(overall_iqr, 4) if overall_iqr is not None and pd.notna(overall_iqr) else None,
        'total_games':    total_n,
        'active_stars':   active_stars,
        'opp_pace':       opp_pace,
        'spread':         spread,
        'home_away':      home_away,
    }

scenarios = player_scenarios(df, 'Mikal Bridges', 'MIN')
scenarios

{'player': 'Mikal Bridges',
 'stat': 'MIN',
 'overall_median': np.float64(35.7),
 'overall_iqr': np.float64(6.7158),
 'total_games': 187,
 'active_stars': {0: {'median': np.float64(0.2417),
   'shrunk_median': np.float64(29.7903),
   'delta': np.float64(-5.9097),
   'hit_rate_vs_overall': np.float64(0.0),
   'n': 2},
  1: {'median': np.float64(37.8183),
   'shrunk_median': np.float64(36.4061),
   'delta': np.float64(0.7061),
   'hit_rate_vs_overall': np.float64(0.8),
   'n': 5},
  2: {'median': np.float64(35.7575),
   'shrunk_median': np.float64(35.7476),
   'delta': np.float64(0.0476),
   'hit_rate_vs_overall': np.float64(0.5208),
   'n': 48},
  3: {'median': np.float64(35.5167),
   'shrunk_median': np.float64(35.5296),
   'delta': np.float64(-0.1704),
   'hit_rate_vs_overall': np.float64(0.4924),
   'n': 132}},
 'opp_pace': {'high_pace': {'median': np.float64(35.1417),
   'shrunk_median': np.float64(35.3109),
   'delta': np.float64(-0.3891),
   'hit_rate_vs_overall': np.float64(0.434